## Code for performing inference with ViFi-CLIP on custom videos

### Please set the corresponding values in the cell below. Afterwards, just run the cells for inference with ViFi-CLIP model

In [30]:
from pathlib import Path

### Set values here ###
config_path = '../data/movyon/config_vificlip_with_freeze.yml'
output_folder_name = ".output"
Path(output_folder_name).mkdir(exist_ok=True)
pretrained_model_path = "/home/zanella/cosmico/ViFi-CLIP/exp/best.pth" # "../best.pth"

# List the action names for which ViFi-CLIP will perform action recognition
class_names = ['accident_stopped_vehicle', 'fire', 'fog', "no_event", "traffic_jam"]
class_labels = ["ACCIDENT", "FIRE", "FOG", "NO_EVENT", "TRAFFIC"]

# Load your video example:

### Import libraries 

In [31]:
import torch
from torch import nn
from utils.config import get_config
from utils.logger import create_logger
import time
import numpy as np
from trainers import vificlip
from datasets.pipeline import Compose

### Setting up configuration, no need to change anything.

In [32]:
# Step 1:
# Configuration class 
class parse_option():
    def __init__(self):
        self.config = config_path
        self.output =  output_folder_name   # Name of output folder to store logs and save weights
        self.resume = pretrained_model_path
        # No need to change below args.
        self.only_test = True
        self.opts = None
        self.batch_size = None
        self.pretrained = None
        self.accumulation_steps = None
        self.local_rank = 0
args = parse_option()
config = get_config(args)
# logger
logger = create_logger(output_dir=args.output, name=f"{config.MODEL.ARCH}")
logger.info(f"working dir: {config.OUTPUT}")

=> merge config from ../data/movyon/config_vificlip_with_freeze.yml
[2025-07-11 15:41:00 ViT-B/16](46959454.py 19): INFO working dir: .output


### Loading ViFi-CLIP and its pretrained weights

In [33]:
# Step 2:
# Create the ViFi-CLIP models and load pretrained weights
model = vificlip.returnCLIP(config,
                            logger=logger,
                            class_names=class_names,)

[2025-07-11 15:41:02 ViT-B/16](vificlip.py 200): INFO Loading CLIP (backbone: ViT-B/16)
Weights not found for some missing keys:  ['transformer.resblocks.0.attn_mask', 'transformer.resblocks.1.attn_mask', 'transformer.resblocks.2.attn_mask', 'transformer.resblocks.3.attn_mask', 'transformer.resblocks.4.attn_mask', 'transformer.resblocks.5.attn_mask', 'transformer.resblocks.6.attn_mask', 'transformer.resblocks.7.attn_mask', 'transformer.resblocks.8.attn_mask', 'transformer.resblocks.9.attn_mask', 'transformer.resblocks.10.attn_mask', 'transformer.resblocks.11.attn_mask']
[2025-07-11 15:41:04 ViT-B/16](vificlip.py 203): INFO Building ViFi-CLIP CLIP
[2025-07-11 15:41:04 ViT-B/16](vificlip.py 238): INFO Freezing all but the last two blocks of the Vision Transformer and the text encoder.
[2025-07-11 15:41:04 ViT-B/16](vificlip.py 249): INFO Unfreezing image_encoder.transformer.resblocks[10].attn.in_proj_weight
[2025-07-11 15:41:04 ViT-B/16](vificlip.py 249): INFO Unfreezing image_encoder.tr

In [34]:
from collections import OrderedDict


logger.info(f"==============> Resuming form {config.MODEL.RESUME}....................")
checkpoint = torch.load(config.MODEL.RESUME, map_location='cpu', weights_only=False)
load_state_dict = checkpoint['model']

## We actually want that the text embeddings are transferred from the finetuned weights:
## There's no point in loading the finetuning with different class prompts
# now remove the unwanted keys:
# if "module.prompt_learner.token_prefix" in load_state_dict:
#     del load_state_dict["module.prompt_learner.token_prefix"]

# if "module.prompt_learner.token_suffix" in load_state_dict:
#     del load_state_dict["module.prompt_learner.token_suffix"]

# if "module.prompt_learner.complete_text_embeddings" in load_state_dict:
#     del load_state_dict["module.prompt_learner.complete_text_embeddings"]

# create new OrderedDict that does not contain `module.`
new_state_dict = OrderedDict()
for k, v in load_state_dict.items():
    name = k[7:] # remove `module.`
    new_state_dict[name] = v


# load params
msg = model.load_state_dict(new_state_dict, strict=False)
logger.info(f"resume model: {msg}")

[2025-07-11 15:41:04 ViT-B/16](2282975454.py 4): INFO ==============> Resuming form /home/zanella/cosmico/ViFi-CLIP/exp/best.pth....................
[2025-07-11 15:41:05 ViT-B/16](2282975454.py 29): INFO resume model: <All keys matched successfully>


### Crete ONNX version

In [6]:
from torch.export import Dim

input = (torch.randn(2, 32, 3, 224, 224),)
onnx_program = torch.onnx.export(
    model.eval().float().to("cpu"),
    input,
    "model.onnx",
    input_names=["video"],
    output_names=["logits"],
    dynamic_shapes=[
        (Dim("batch"), Dim("frames"), Dim.STATIC, Dim.STATIC, Dim.STATIC)
    ],
    dynamo=True
)


[torch.onnx] Obtain model graph for `ViFiCLIP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ViFiCLIP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 18 of general pattern rewrite rules.


In [27]:
import numpy as np
import onnx
from onnxconverter_common.auto_mixed_precision import auto_convert_mixed_precision
from onnxconverter_common import float16


onnx_model = onnx.load("model.onnx")

onnx_model_fp16 = float16.convert_float_to_float16(onnx_model, keep_io_types=True)
onnx.save(onnx_model_fp16, "model_fp16.onnx")

onnx_model_amp = auto_convert_mixed_precision(onnx_model, 
                                              {"video": np.random.randn(1, 32, 3, 224, 224).astype(np.float32)}, 
                                              keep_io_types=True, rtol=0.01, atol=0.001)
onnx.save(onnx_model_amp, "model_amp.onnx")


/home/zanella/cosmico/ViFi-CLIP/.pixi/envs/dev/lib/python3.12/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 3.91741181715588e-08 will be truncated to 1e-07
  warnings.warn(
/home/zanella/cosmico/ViFi-CLIP/.pixi/envs/dev/lib/python3.12/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 5.960464477539063e-08 will be truncated to 1e-07
  warnings.warn(
/home/zanella/cosmico/ViFi-CLIP/.pixi/envs/dev/lib/python3.12/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -5.960464477539063e-08 will be truncated to -1e-07
  warnings.warn(
/home/zanella/cosmico/ViFi-CLIP/.pixi/envs/dev/lib/python3.12/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 4.492801863875684e-08 will be truncated to 1e-07
  warnings.warn(
/home/zanella/cosmico/ViFi-CLIP/.pixi/envs/dev/lib/python3.12/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -2.528398157508

Adding missing dtypes for 0 outputs
Running attempt 1 excluding conversion of 0 nodes
[]


2025-07-11 15:40:07.670075589 [W:onnxruntime:, constant_folding.cc:278 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Add node 'node_Add_1326'
2025-07-11 15:40:07.697822483 [W:onnxruntime:, constant_folding.cc:278 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Add node 'node_Add_1326'


True
Attempt succeeded.
[*1149*]
Done: []
[]


2025-07-11 15:40:18.780629072 [W:onnxruntime:, constant_folding.cc:278 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Add node 'node_Add_1326'
2025-07-11 15:40:18.805328883 [W:onnxruntime:, constant_folding.cc:278 ApplyImpl] Could not find a CPU kernel and hence can't constant fold Add node 'node_Add_1326'


Final model validated successfully.


In [35]:
import onnx
import torch
import onnxruntime

onnx_model = onnx.load("model.onnx")
onnx.checker.check_model(onnx_model)

ort_session = onnxruntime.InferenceSession(
    "model.onnx", providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)

In [40]:
import time

video = torch.randn(1, 32, 3, 224, 224)
s_onnx = time.time()
ort_logits = ort_session.run(["logits"], {"video": video.numpy()})[0]
t_onnx = time.time() - s_onnx

model.float().cuda().eval()
s_torch = time.time()
with torch.no_grad():
    logits = model(video.cuda())
t_torch = time.time() - s_torch

print("ONNX:", t_onnx)
print(ort_logits)
print("Torch:", t_torch)
print(logits.numpy(force=True))
print("Equal:", np.allclose(ort_logits, logits.numpy(force=True)))

ONNX: 0.052591800689697266
[[19.57317  19.776289 19.722956 20.33857  19.690193]]
Torch: 0.08758115768432617
[[19.572472 19.775553 19.72219  20.338005 19.689342]]
Equal: False


In [ ]:
ort_session = onnxruntime.InferenceSession(
    "model_fp16.onnx", providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)

s_onnx = time.time()
ort_logits = ort_session.run(["logits"], {"video": video.numpy()})[0]
t_onnx = time.time() - s_onnx

print(ort_logits)
print("ONNX:", t_onnx)

[[19.578125 19.78125  19.734375 20.34375  19.703125]]
ONNX: 0.03261995315551758


### Preprocessing input video 

In [10]:
# Step 3: 
# Preprocessing for video
img_norm_cfg = dict(
    mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375], to_bgr=False)
scale_resize = int(256 / 224 * config.DATA.INPUT_SIZE)
val_pipeline = [
    dict(type='DecordInit'),
    dict(type='SampleFrames', clip_len=1, frame_interval=1, num_clips=config.DATA.NUM_FRAMES, test_mode=True),
    dict(type='DecordDecode'),
    dict(type='Resize', scale=(-1, 224)),
    dict(type='CenterCrop', crop_size=config.DATA.INPUT_SIZE),
    dict(type='Normalize', **img_norm_cfg),
    dict(type='FormatShape', input_format='NCHW'),
    dict(type='Collect', keys=['imgs'], meta_keys=[]),
    dict(type='ToTensor', keys=['imgs'])
]
if config.TEST.NUM_CROP == 3:
    val_pipeline[3] = dict(type='Resize', scale=(-1, config.DATA.INPUT_SIZE))
    val_pipeline[4] = dict(type='ThreeCrop', crop_size=config.DATA.INPUT_SIZE)
if config.TEST.NUM_CLIP > 1:
    val_pipeline[1] = dict(type='SampleFrames', clip_len=1, frame_interval=1, num_clips=config.DATA.NUM_FRAMES, multiview=config.TEST.NUM_CLIP)

val_pipeline[1] = dict(type='SampleFrames', clip_len=32, frame_interval=1, num_clips=1, test_mode=True)

pipeline = Compose(val_pipeline)

### ViFi-CLIP inference with given video

In [8]:
# Inference code with torch transformations

from pathlib import Path
from datetime import datetime
import math
import yaml
import csv

try:
    import torchvision.transforms.v2 as T
except ModuleNotFoundError:
    import torchvision.transforms as T
from torchvision.ops import Permute
import decord
from decord import VideoReader
from tqdm import trange

# decord.bridge.set_bridge("torch")

model = model.float().cuda()  # changing to cuda here

video_dir = Path("../dataset_videos/videos")
root_path = Path("../.outputs/vificlip_results")
exp_dir = root_path / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
preds_dir = exp_dir / "predictions"
preds_dir.mkdir(parents=True, exist_ok=True)

# Normalization values
normalize = T.Normalize(mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375])

# Resize and crop
transform = T.Compose([
    Permute([0, 3, 1, 2]),
    T.Resize(config.DATA.INPUT_SIZE),             # Resize shorter side to scale_resize
    T.CenterCrop(config.DATA.INPUT_SIZE),         # Center crop to final size
    normalize                                     # Normalize with given mean/std
])

videos_predictions = []

with open(exp_dir / "config.yml", mode="w") as file:
        yaml.dump(dict(zip(class_labels, class_names)), file)

video_paths = sorted(video_dir.glob("*.mp4"))
clip_len = config.DATA.NUM_FRAMES  # 32
for video_path in video_paths:
    reader = VideoReader(str(video_path.resolve()))

    n_clips = int(math.floor(len(reader) / clip_len)) # Use ceil to include last frames
    print(f"Processing video: {video_path}: {n_clips} clips found.")

    clip_logits = []
    clip_predictions = []  # Store frame ranges for each clip
    for idx_clip in trange(n_clips):
        start_frame = idx_clip * clip_len
        end_frame = (idx_clip + 1) * clip_len
        video_clip = reader[start_frame:end_frame].asnumpy()
        video_clip = torch.from_numpy(video_clip).float()
        video_clip = transform(video_clip)
        video_clip = video_clip.unsqueeze(0).cuda()
        # Inference through ViFi-CLIP
        with torch.no_grad():
            with torch.autocast("cuda"):
            # model.half()
            # video_clip = video_clip.half()
                logits = model(video_clip)
        max_index = logits.argmax()
        label = class_labels[max_index]
        clip_logits.append(logits)
        # Calculate frame range for the clip
        clip_predictions.append((start_frame, end_frame, label))

    # Concatenate logits for all clips
    # Extract the max logits and their indices
    logits = torch.cat(clip_logits, dim=0)
    mean_logits = logits.mean(dim=0)
    print("Mean logits:", mean_logits)
    predicted_label = class_labels[mean_logits.argmax()]

    max_indices = logits.argmax(dim=1)
    mode_index = max_indices.mode()[0]
    predicted_mode_label = class_labels[mode_index]
    print("Average result:", predicted_label)
    print("Mode result:", predicted_mode_label)
    del reader
    torch.cuda.empty_cache()

    videos_predictions.append((video_path.name, predicted_label))

    # Save the predictions
    with open(preds_dir / video_path.with_suffix(".csv").name, mode="w") as file:
        writer = csv.writer(file)
        writer.writerow(["start", "end", "caption"])
        writer.writerows(clip_predictions)

with open(exp_dir / "labels.csv", mode="w") as file:
    writer = csv.writer(file)
    writer.writerow(["example", "prediction"])
    writer.writerows(videos_predictions)

# [20.3750, 20.6875, 20.2500, 21.8125, 19.2812] versione torch precedente

Processing video: ../dataset_videos/videos/994_mezzoFermo.mp4: 131 clips found.


100%|██████████| 131/131 [00:29<00:00,  4.41it/s]


Mean logits: tensor([19.7344, 18.0938, 19.8438, 20.6562, 19.0156], device='cuda:0',
       dtype=torch.float16)
Average result: NO_EVENT
Mode result: NO_EVENT
Processing video: ../dataset_videos/videos/fireA10_Km025.7_T11064_D20240704_H172023_N0.mp4: 98 clips found.


100%|██████████| 98/98 [00:07<00:00, 13.11it/s]


Mean logits: tensor([18.9219, 26.2031, 19.3438, 19.2656, 18.6562], device='cuda:0',
       dtype=torch.float16)
Average result: FIRE
Mode result: FIRE
Processing video: ../dataset_videos/videos/fire_A10_Km025.7_T11064_D20240704_H181007_N4.mp4: 105 clips found.


...

Mean logits: tensor([20.5156, 17.9688, 21.0312, 21.1250, 20.2188], device='cuda:0',
       dtype=torch.float16)
Average result: NO_EVENT
Mode result: NO_EVENT
Processing video: ../dataset_videos/videos/traffic_jamA26_Km025.0_T11063_D20240817_H103720_N9.mp4: 105 clips found.


100%|██████████| 105/105 [00:07<00:00, 13.18it/s]


Mean logits: tensor([17.9375, 17.3438, 17.5000, 19.9062, 19.4844], device='cuda:0',
       dtype=torch.float16)
Average result: NO_EVENT
Mode result: NO_EVENT
Processing video: ../dataset_videos/videos/traffic_jamA26_Km030.0_T11078_D20240803_H081939_N1_Codea_tratti.mp4: 109 clips found.


100%|██████████| 109/109 [00:04<00:00, 21.86it/s]

Mean logits: tensor([16.1719, 15.7578, 15.6797, 18.4219, 20.2031], device='cuda:0',
       dtype=torch.float16)
Average result: TRAFFIC
Mode result: TRAFFIC


Output dei inferenza con versione precedente:
```
Processing video: ../dataset_videos/videos/994_mezzoFermo.mp4: 131 clips found.
100%|██████████| 131/131 [00:26<00:00,  4.92it/s]
Mean logits: tensor([19.1250, 20.8438, 21.6406, 24.8750, 25.1406], device='cuda:0',
       dtype=torch.float16)
Average result: ACCIDENT
Mode result: ACCIDENT
Processing video: ../dataset_videos/videos/fireA10_Km025.7_T11064_D20240704_H172023_N0.mp4: 98 clips found.
100%|██████████| 98/98 [00:05<00:00, 17.93it/s]
Mean logits: tensor([25.7188, 21.2812, 21.0000, 22.5781, 22.9531], device='cuda:0',
       dtype=torch.float16)
Average result: FIRE
Mode result: FIRE
Processing video: ../dataset_videos/videos/fire_A10_Km025.7_T11064_D20240704_H181007_N4.mp4: 105 clips found.
100%|██████████| 105/105 [00:06<00:00, 15.58it/s]
Mean logits: tensor([24.9688, 21.1250, 21.7969, 23.2656, 24.0000], device='cuda:0',
       dtype=torch.float16)
Average result: FIRE
Mode result: FIRE
Processing video: ../dataset_videos/videos/fog17874496-uhd_2560_1440_30fps.mp4: 72 clips found.
100%|██████████| 72/72 [00:14<00:00,  4.85it/s]
Mean logits: tensor([22.0469, 28.2656, 21.4844, 23.9844, 24.0312], device='cuda:0',
       dtype=torch.float16)
Average result: FOG
Mode result: FOG
Processing video: ../dataset_videos/videos/fog18618213-uhd_3840_2160_30fps.mp4: 79 clips found.
100%|██████████| 79/79 [00:15<00:00,  5.16it/s]
Mean logits: tensor([19.8906, 25.3750, 21.4844, 24.5781, 22.9375], device='cuda:0',
       dtype=torch.float16)
Average result: FOG
Mode result: FOG
Processing video: ../dataset_videos/videos/fog4763084-uhd_3840_2160_24fps.mp4: 102 clips found.
100%|██████████| 102/102 [00:22<00:00,  4.45it/s]
Mean logits: tensor([19.7812, 27.4688, 22.4062, 21.1719, 19.3438], device='cuda:0',
       dtype=torch.float16)
Average result: FOG
Mode result: FOG
Processing video: ../dataset_videos/videos/fog_2560897-uhd_3840_2160_30fps.mp4: 201 clips found.
 54%|█████▍    | 109/201 [00:25<00:12,  7.18it/s]
 ```

In [11]:
# Original code for
from pathlib import Path

model = model.float().cuda()  # changing to cuda here

video_dir = Path("../dataset_videos/videos")

video_paths = sorted(video_dir.glob("*.mp4"))
for video_path in video_paths:
    dict_file = {'filename': str(video_path), 'tar': False, 'modality': 'RGB', 'start_index': 0}
    video = pipeline(dict_file)
    video_tensor = video['imgs'].unsqueeze(0).cuda().float()
    print(video_tensor.shape)
    # Inference through ViFi-CLIP
    with torch.no_grad():
        with torch.autocast("cuda"):
            logits = model(video_tensor)
    print(logits)
    pred_index = logits.argmax(-1)
    print(class_names[pred_index])

# [20.3750, 20.6875, 20.2500, 21.8125, 19.2812]

torch.Size([1, 32, 3, 224, 224])
tensor([[20.4844, 20.4844, 19.3594, 21.5469, 18.7500]], device='cuda:0',
       dtype=torch.float16)
no_event
torch.Size([1, 32, 3, 224, 224])
tensor([[18.2188, 27.3594, 19.3750, 19.0312, 18.1094]], device='cuda:0',
       dtype=torch.float16)
fire
torch.Size([1, 32, 3, 224, 224])
tensor([[18.5625, 25.9062, 19.4688, 19.2969, 18.5156]], device='cuda:0',
       dtype=torch.float16)
fire
torch.Size([1, 32, 3, 224, 224])
tensor([[16.7188, 20.0625, 23.2188, 19.5000, 18.7812]], device='cuda:0',
       dtype=torch.float16)
fog
torch.Size([1, 32, 3, 224, 224])
tensor([[16.8594, 19.9219, 22.4688, 20.4844, 19.5469]], device='cuda:0',
       dtype=torch.float16)
fog
torch.Size([1, 32, 3, 224, 224])
tensor([[16.8438, 18.8438, 22.7344, 19.5625, 19.5312]], device='cuda:0',
       dtype=torch.float16)
fog
torch.Size([1, 32, 3, 224, 224])
tensor([[15.6875, 18.4531, 21.1250, 18.5156, 18.3906]], device='cuda:0',
       dtype=torch.float16)
fog
torch.Size([1, 32, 3, 224, 

KeyboardInterrupt: 

### Esperimenti sulla pipeline

In [27]:
from pathlib import Path
from decord import VideoReader
from torchvision.io import write_video


scale_resize = int(256 / 224 * config.DATA.INPUT_SIZE)
img_norm_cfg = dict(
    mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375], to_bgr=False)

# Train pipeline
train_pipeline = [
    dict(type='DecordInit'),
    dict(type='SampleFrames', clip_len=1, frame_interval=1, num_clips=config.DATA.NUM_FRAMES),
    dict(type='DecordDecode'),
    dict(type='Resize', scale=(-1, scale_resize)),
    dict(
        type='MultiScaleCrop',
        input_size=config.DATA.INPUT_SIZE,
        scales=(1, 0.875, 0.75, 0.66),
        random_crop=False,
        max_wh_scale_gap=1),
    dict(type='Resize', scale=(config.DATA.INPUT_SIZE, config.DATA.INPUT_SIZE), keep_ratio=False),
    dict(type='Flip', flip_ratio=0.5),
    dict(type='ColorJitter', p=config.AUG.COLOR_JITTER),
    dict(type='GrayScale', p=config.AUG.GRAY_SCALE),
    # dict(type='Normalize', **img_norm_cfg),
    dict(type='FormatShape', input_format='NCHW'),
    dict(type='Collect', keys=['imgs'], meta_keys=[]),
    dict(type='ToTensor', keys=['imgs']),
]

train_pipeline = Compose(train_pipeline)

# Val pipeline
val_pipeline = [
    dict(type='DecordInit'),
    dict(type='SampleFrames', clip_len=1, frame_interval=1, num_clips=config.DATA.NUM_FRAMES, test_mode=True),
    dict(type='DecordDecode'),
    dict(type='Resize', scale=(-1, 224)),
    dict(type='CenterCrop', crop_size=config.DATA.INPUT_SIZE),
    dict(type='Normalize', **img_norm_cfg),
    dict(type='FormatShape', input_format='NCHW'),
    dict(type='Collect', keys=['imgs'], meta_keys=[]),
    dict(type='ToTensor', keys=['imgs'])
]
if config.TEST.NUM_CROP == 3:
    val_pipeline[3] = dict(type='Resize', scale=(-1, config.DATA.INPUT_SIZE))
    val_pipeline[4] = dict(type='ThreeCrop', crop_size=config.DATA.INPUT_SIZE)
if config.TEST.NUM_CLIP > 1:
    val_pipeline[1] = dict(type='SampleFrames', clip_len=1, frame_interval=1, num_clips=config.DATA.NUM_FRAMES, multiview=config.TEST.NUM_CLIP)

val_pipeline = Compose(val_pipeline)

model = model.float().cuda()  # changing to cuda here

video_dir = Path("../dataset_videos/videos")
save_dir = Path("train-pipeline")
save_dir.mkdir(exist_ok=True)

video_paths = sorted(video_dir.glob("*.mp4"))
for video_path in video_paths:
    dict_file = {'filename': str(video_path), 'tar': False, 'modality': 'RGB', 'start_index': 0}
    video = train_pipeline(dict_file)
    reader = VideoReader(str(video_path))
    video_tensor = video['imgs'].permute(0, 2, 3, 1)
    print(video_tensor.shape)
    fps = int(round(reader.get_avg_fps()))
    print("fps:", fps)
    write_video(str(save_dir / video_path.name), video_tensor, fps=5,
            video_codec="h264",
            options={
                "crf": "18",          # Lower = better quality (18 is visually lossless)
                "preset": "slow"      # Slow = better compression and quality
            }
    )
